# RAPTOR：DeepSeek + 百炼 API Demo

保留官方 `demo.ipynb` 和 RAPTOR 源码，仅通过三个基类适配 API。
默认摘要和问答使用 DeepSeek，向量使用百炼 `text-embedding-v4`（1024 维）；型号和维度均可通过环境变量切换。
分块、UMAP/GMM 聚类和检索继续在本地执行；不加载本地神经网络模型。

先在项目根目录运行：
```bash
uv pip install --python .venv/bin/python -r requirements-cus.txt
```
在 IDE 中选择本项目 `.venv/bin/python` 内核（Python 3.10）。

根目录 `.env` 仅保存非密钥配置，格式参见 `.env.example`：

| 环境变量 | 用途 |
| --- | --- |
| `DEEPSEEK_SUMMARIZATION_MODEL` | 摘要模型 |
| `DEEPSEEK_QA_MODEL` | 问答模型 |
| `DASHSCOPE_EMBEDDING_MODEL` | 向量模型 |
| `DASHSCOPE_EMBEDDING_DIMENSIONS` | 向量维度（必须与模型匹配） |
| `DEEPSEEK_QA_MAX_TOKENS` | 答案输出长度上限 |
| `DEEPSEEK_BASE_URL` / `DASHSCOPE_BASE_URL` | 两个服务的兼容接口地址 |

`DEEPSEEK_API_KEY`、`DASHSCOPE_API_KEY` 从 shell 导出的环境变量继承，不写进 `.env` 或 Notebook。
在已加载 `.bashrc` 的终端中启动 IDE/Notebook，使内核继承密钥；本代码不会读取或执行 `.bashrc`。
已有进程环境变量优先于 `.env`；改动 `.env` 后重启内核再执行，避免继续使用已加载的旧值。
构造函数显式参数优先于环境变量。`DEEPSEEK_MODEL` 可作摘要/问答的共享回退，单独指定的用途模型优先。
切换型号仍须使用同类的兼容 API：DeepSeek Chat Completions 和百炼文本稠密向量接口。
现有 `.gitignore` 已忽略 `.env` 和 `local/`。百炼地址需与地域、业务空间、API Key 匹配，
并以 `/compatible-mode/v1` 结尾。

官方文档：[DeepSeek](https://api-docs.deepseek.com/)、
[非思考模式](https://api-docs.deepseek.com/guides/thinking_mode/)、
[百炼 Embedding 参数与地域地址](https://help.aliyun.com/zh/model-studio/text-embedding-synchronous-api)。
已核实 v4 支持每条 8192 Token、每批最多 10 条；本适配器每次传入一条文本。

In [1]:
import hashlib
import json
import os
from pathlib import Path
import sys

from dotenv import load_dotenv

ROOT = Path.cwd().resolve()
if not (ROOT / "requirements-cus.txt").exists():
    raise RuntimeError("请从 RAPTOR 项目根目录打开本 Notebook。")
load_dotenv(ROOT / ".env", override=False)
# 仅首次需要下载分词数据；不是模型权重，也不产生 API 费用。
os.environ.setdefault("TIKTOKEN_CACHE_DIR", str(ROOT / "local/tiktoken-cache"))
os.environ.setdefault("HF_HOME", str(ROOT / "local/huggingface"))
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

print("Python:", sys.version.split()[0])
print("解释器:", sys.executable)
required_env = ("DEEPSEEK_API_KEY", "DASHSCOPE_API_KEY", "DASHSCOPE_BASE_URL")
missing_env = [name for name in required_env if not os.getenv(name, "").strip()]
print("配置检查:", "齐全" if not missing_env else "缺少 " + ", ".join(missing_env))
# 不打印环境变量的值。

Python: 3.10.22
解释器: /home/ryan/projects/raptor/.venv/bin/python
配置检查: 齐全


In [2]:
from raptor import RetrievalAugmentation, RetrievalAugmentationConfig
from raptor.SummarizationModels_cus import DeepSeekSummarizationModel
from raptor.QAModels_cus import DeepSeekQAModel
from raptor.EmbeddingModels_cus import BailianEmbeddingModel
from raptor.utils import split_text
import tiktoken

text = (ROOT / "demo/sample.txt").read_text(encoding="utf-8")
tokenizer = tiktoken.get_encoding("cl100k_base")    # 检查缓存，不存在的时候下载分词数据；utils里也有这个逻辑
chunks = split_text(text, tokenizer, max_tokens=100)
print("Cinderella 文本字符数:", len(text))
print("预计叶子节点数:", len(chunks))
print(text[:100])

/home/ryan/projects/raptor/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-10-25 00:01:23,765 - Loading faiss with AVX512 support.
2026-10-25 00:01:23,766 - Could not load library with AVX512 support due to:
ModuleNotFoundError("No module named 'faiss.swigfaiss_avx512'")
2026-10-25 00:01:23,767 - Loading faiss with AVX2 support.
2026-10-25 00:01:23,767 - Could not load library with AVX2 support due to:
ModuleNotFoundError("No module named 'faiss.swigfaiss_avx2'")
2026-10-25 00:01:23,768 - Loading faiss.
2026-10-25 00:01:24,058 - Successfully loaded faiss.


Cinderella 文本字符数: 14025
预计叶子节点数: 35
The wife of a rich man fell sick, and as she felt that her end
was drawing near, she called her only


上述单元不会调用模型 API。可先在终端运行免费测试：
```bash
.venv/bin/python -m unittest discover -s tests -v
```
测试使用模拟 HTTP 响应、真实 RAPTOR 建树/检索逻辑，并阻止网络连接。

以下单元包含付费调用。将 `RUN_PAID_API` 改为 `True` 执行完整流程。
首次建议保留最小连通性测试；之后可关闭，避免重复付费。
完整建树会为每个叶子和摘要节点生成向量，并为每个父节点调用摘要 API。

In [5]:
RUN_PAID_API = True
RUN_API_SMOKE_TEST = True
REBUILD_TREE = False
RUN_RELOADED_QA = False  # 改为 True 将额外进行一次检索向量和问答调用。
QUESTION = "How did Cinderella reach her happy ending?"
SAVE_PATH = ROOT / "local/cinderella-cus.pkl"
META_PATH = SAVE_PATH.with_suffix(".json")

if RUN_PAID_API and missing_env:
    raise RuntimeError("缺少配置：" + ", ".join(missing_env))

def make_custom_config():
    return RetrievalAugmentationConfig(
        summarization_model=DeepSeekSummarizationModel(),
        qa_model=DeepSeekQAModel(),
        embedding_model=BailianEmbeddingModel(),
    )
# 模型和 API 参数由环境变量读取；不覆盖算法参数：100-token 分块、最多 5 层、100-token 摘要、UMAP 10 维。

先测试三个接口，只使用很短的公开示例文本。客户端只对临时故障进行有限重试，
鉴权错误直接失败。输出不包含密钥或完整 API 响应。

In [6]:
if RUN_PAID_API:
    custom_config = make_custom_config()
    summarizer = custom_config.tree_builder_config.summarization_model
    embedding = custom_config.tree_retriever_config.embedding_model
    qa = custom_config.qa_model
    if RUN_API_SMOKE_TEST:
        short_context = "Cinderella married the prince after the glass slipper fit her foot."
        vector = embedding.create_embedding(short_context)
        summary = summarizer.summarize(short_context, max_tokens=60)
        smoke_answer = qa.answer_question(short_context, "Who did Cinderella marry?")
        print("Embedding 维度:", len(vector))
        print("摘要接口:", summary)
        print("问答接口:", smoke_answer)
else:
    print("免费检查模式：跳过模型 API 调用。")

2026-10-25 00:08:40,731 - HTTP Request: POST https://llm-q0muet2tfzluckwm.cn-beijing.maas.aliyuncs.com/compatible-mode/v1/embeddings "HTTP/1.1 200 OK"
2026-10-25 00:08:40,905 - HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"
2026-10-25 00:08:42,116 - HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"


Embedding 维度: 1024
摘要接口: Cinderella married the prince after the glass slipper fit her foot. The key details are: the glass slipper was used to identify her, it fit her foot, and the marriage to the prince followed.
问答接口: The prince


构建或加载自定义树。索引保存在被 Git 忽略的 `local/` 中，
并记录文本哈希、建树模型、接口地址和分块配置；配置不匹配时需显式设置 `REBUILD_TREE=True`。
仅切换问答模型或答案长度不会要求重新建树；切换摘要/向量模型、向量维度或建树接口地址则需要重建。
不会复用官方 `demo/cinderella`。

In [ ]:
if RUN_PAID_API:
    fingerprint = {
        "text_sha256": hashlib.sha256(text.encode("utf-8")).hexdigest(),
        "summarization_model": summarizer.model,
        "summarization_base_url": str(summarizer.client.base_url),
        "embedding_base_url": str(embedding.client.base_url),
        "embedding_model": embedding.model,
        "dimensions": embedding.dimensions,
        "chunk_tokens": 100,
        "max_layers": 5,
        "summary_tokens": 100,
        "reduction_dimension": 10,
    }
    if SAVE_PATH.exists() and not REBUILD_TREE:
        if not META_PATH.exists() or json.loads(META_PATH.read_text()) != fingerprint:
            raise RuntimeError("索引元数据不匹配，请设置 REBUILD_TREE=True 后重建。")
        RA = RetrievalAugmentation(config=custom_config, tree=str(SAVE_PATH))   # 会触发日志
        print("已加载自定义索引。")
    else:
        RA = RetrievalAugmentation(config=custom_config)    # 会触发日志
        RA.add_documents(text)
        SAVE_PATH.parent.mkdir(parents=True, exist_ok=True)
        RA.save(str(SAVE_PATH))
        META_PATH.write_text(json.dumps(fingerprint, indent=2), encoding="utf-8")
        print("已构建并保存自定义索引。")
    print("各层节点数:", {layer: len(nodes) for layer, nodes in RA.tree.layer_to_nodes.items()})
    print("总节点数:", len(RA.tree.all_nodes))

2026-10-25 00:13:54,257 - Successfully initialized TreeBuilder with Config 
        TreeBuilderConfig:
            Tokenizer: <Encoding 'cl100k_base'>
            Max Tokens: 100
            Num Layers: 5
            Threshold: 0.5
            Top K: 5
            Selection Mode: top_k
            Summarization Length: 100
            Summarization Model: <raptor.SummarizationModels_cus.DeepSeekSummarizationModel object at 0x7030e589b790>
            Embedding Models: {'EMB': <raptor.EmbeddingModels_cus.BailianEmbeddingModel object at 0x7030e60b1e70>}
            Cluster Embedding Model: EMB
        
        Reduction Dimension: 10
        Clustering Algorithm: RAPTOR_Clustering
        Clustering Parameters: {}
        
2026-10-25 00:13:54,258 - Successfully initialized ClusterTreeBuilder with Config 
        TreeBuilderConfig:
            Tokenizer: <Encoding 'cl100k_base'>
            Max Tokens: 100
            Num Layers: 5
            Threshold: 0.5
            Top K: 5
         

已加载自定义索引。
各层节点数: {0: 35, 1: 7}
总节点数: 42


检索保留官方默认：折叠树检索、`top_k=10`、上下文最多 3500 Token。
这一步会调用一次百炼 API 生成问题向量。

In [8]:
if RUN_PAID_API:
    context, layer_information = RA.retrieve(
        question=QUESTION,
        top_k=10,
        max_tokens=3500,
        collapse_tree=True,
        return_layer_information=True,
    )
    print("检索上下文：")
    print(context)
    print("检索节点层级：", layer_information)

2026-10-25 00:10:23,489 - Using collapsed_tree
2026-10-25 00:10:23,598 - HTTP Request: POST https://llm-q0muet2tfzluckwm.cn-beijing.maas.aliyuncs.com/compatible-mode/v1/embeddings "HTTP/1.1 200 OK"


检索上下文：
A man preparing to travel asked his two step-daughters what gifts he should bring back for them. The first asked for beautiful dresses, and the second asked for pearls and jewels. When he asked Cinderella what she wanted, she asked him to break off for her the first branch that knocked against his hat on his way home.  He bought the beautiful dresses and the pearls and jewels for his step-daughters. On his way home, as he rode through a green thicket, a hazel tw

a dim little oil-lamp was burning on the mantle-piece, for cinderella had jumped quickly down from the back of the pigeon-house and had run to the little hazel-tree, and there she had taken off her beautiful clothes and laid them on the grave, and the bird had taken them away again, and then she had seated herself in the kitchen amongst the ashes in her grey gown Next day when the festival began afresh, and her parents and

The passage summarizes a version of the Cinderella story. A prince approaches the heroine, takes 

将刚检索的上下文传给问答模型，避免重复生成问题向量。
也可使用 `RA.answer_question(QUESTION)` 一次完成检索与问答。
注意：该方法的 `max_tokens` 限制检索上下文，答案生成上限由 `DEEPSEEK_QA_MAX_TOKENS` 配置。

In [9]:
if RUN_PAID_API:
    answer = RA.qa_model.answer_question(context, QUESTION)
    print("问题:", QUESTION)
    print("答案:", answer)

2026-10-25 00:10:23,792 - HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"


问题: How did Cinderella reach her happy ending?
答案: Based only on the given context, the passages do not describe how Cinderella ultimately reaches her happy ending. They cover earlier events—her mother's death and burial, her mistreatment by the stepfamily, the hazel tree and the helpful bird, the three-day festival, the prince dancing only with her, her escapes (including into the pigeon-house), and the golden slipper being tried on by the stepsisters—but the context ends before any marriage or final resolution is narrated. Therefore, the context is insufficient to answer how Cinderella reaches her happy ending.


重新加载时继续传入相同的 `custom_config`，确保使用相同的向量模型和维度。
仅加载不会调用 API；可选的重新问答会产生额外调用。

In [10]:
if RUN_PAID_API:
    RA_loaded = RetrievalAugmentation(config=custom_config, tree=str(SAVE_PATH))
    assert len(RA_loaded.tree.all_nodes) == len(RA.tree.all_nodes)
    print("索引重新加载检查通过。")
    if RUN_RELOADED_QA:
        print("重新加载后的答案:", RA_loaded.answer_question(QUESTION))

2026-10-25 00:10:24,802 - Successfully initialized TreeBuilder with Config 
        TreeBuilderConfig:
            Tokenizer: <Encoding 'cl100k_base'>
            Max Tokens: 100
            Num Layers: 5
            Threshold: 0.5
            Top K: 5
            Selection Mode: top_k
            Summarization Length: 100
            Summarization Model: <raptor.SummarizationModels_cus.DeepSeekSummarizationModel object at 0x7030e589b790>
            Embedding Models: {'EMB': <raptor.EmbeddingModels_cus.BailianEmbeddingModel object at 0x7030e60b1e70>}
            Cluster Embedding Model: EMB
        
        Reduction Dimension: 10
        Clustering Algorithm: RAPTOR_Clustering
        Clustering Parameters: {}
        
2026-10-25 00:10:24,802 - Successfully initialized ClusterTreeBuilder with Config 
        TreeBuilderConfig:
            Tokenizer: <Encoding 'cl100k_base'>
            Max Tokens: 100
            Num Layers: 5
            Threshold: 0.5
            Top K: 5
         

索引重新加载检查通过。
